In [1]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.document_loaders import OnlinePDFLoader

In [2]:
local_path="ABSLI_Group_Bima_Yojana_V04_Brochure_Web_Version_1c032b927b.pdf"
if local_path:
    loader=UnstructuredPDFLoader(local_path)
    data=loader.load()
else:
    print("path not found")

In [3]:
data[0].page_content

'Uplifting your family’s ﬁnancial journey with security and stability at every stage of life\n\nAditya Birla Sun Life Insurance Group Bima Yojana A Non-participating, Non-linked life group micro insurance single premium pure risk term plan\n\nINTRODUCING ABSLI GROUP BIMA YOJANA\n\nLife is full of uncertainties. This product has been designed especially for Employer-Employee groups and non-employer-employee groups. ABSLI Group Bima Yojana is a non-linked, non-participating, group term life pure risk micro-insurance single premium product. The aim of this product is to provide protection against ﬁnancial liabilities at a nominal cost in the event of unfortunate death of an Employee/Member.\n\nKEY FEATURES OF THE PLAN\n\nLevel death beneﬁt\n\nSingle life or joint life cover\n\nSingle Pay\n\nPLAN ELIGIBILITY\n\nThe product has been designed for the Employees/Members of:\n\nEmployer-Employee groups • Non-Employer-Employee groups\n\nNon employer-employee homogeneous groups includes:\n\n1. An

VECTOR EMBEDDING

In [4]:
!ollama pull nomic-embed-text

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


In [5]:
!ollama list

NAME                              ID              SIZE      MODIFIED               
nomic-embed-text:latest           0a109f422b47    274 MB    Less than a second ago    
qwen3.5:0.8b                      f3817196d142    1.0 GB    9 minutes ago             
nomic-embed-text-v2-moe:latest    ff9c2f10ef5e    957 MB    58 minutes ago            


In [6]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

In [21]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=text_splitter.split_documents(data)

In [8]:
vector_db=Chroma.from_documents(
    documents=chunks,
    embedding=OllamaEmbeddings(model="nomic-embed-text",show_progress=True),
    collection_name="local_rag"
)

OllamaEmbeddings: 100%|██████████| 2/2 [00:10<00:00,  5.07s/it]


RETRIEVE

In [9]:
from langchain.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnablePassthrough #question
from langchain.retrievers.multi_query import MultiQueryRetriever # to generate mutiple questions

In [15]:
local_model="phi3"
llm=ChatOllama(model=local_model)

In [16]:
QUERY_PROMPT= PromptTemplate(
    input_variables=["question"],
    template=""" You are an AI language model assistant . Your task is to generate five different
     versions of the given user questions to retrieve relevant documents from a vector
      database. By generating multiple perspectives on the user question, your goal is to help the user
       overcome some of the limitations of the distance-based similarity search. Provide these alternative
        questions seperated by newlines.
         Original question: {question} """,
)

In [23]:
retriever= MultiQueryRetriever.from_llm(
    vector_db.as_retriever(search_kwargs={'k':2}),
    llm,
    prompt=QUERY_PROMPT
)
# context is what we get from vector database
template=""" Answer the question based only on the following context:
{context}
Question:{question} """

prompt=ChatPromptTemplate.from_template(template)

In [18]:
chain =(
    {"context":retriever,"question":RunnablePassthrough()}
     | prompt
     | llm
     | StrOutputParser()
)

In [ ]:
import streamlit as st

st.title("📄 Insurance PDF Chatbot")

# chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# show previous messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# user input
if prompt := st.chat_input("Ask something about the document"):

    # show user message
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("user"):
        st.markdown(prompt)

    # generate response
    response = chain.invoke(prompt)

    # show assistant response
    with st.chat_message("assistant"):
        st.markdown(response)

    st.session_state.messages.append({"role": "assistant", "content": response})